#### 4x4 Corridor Scenario (5D Bicycle model)

This notebook follows the `square_3D.ipynb` workflow and implements:
1. A `4x4` map with two rectangular obstacles matching the provided figure.
2. A Temporal Logic Tree task: avoid both obstacles, reach the checkpoint first, and only then reach the goal region.
3. Backward reachable tube (BRT) computation and export to a NumPy file.

State: `[x, y, yaw, delta, vel]` — position, heading, steering angle, speed.


In [ ]:
import sys
import os
import jax

jax.config.update("jax_platform_name", "cpu")

# Add `pyspect/src` to path when running from `pyspect/hjb_mpc`.
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(parent_dir, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

from functools import reduce
from math import pi
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from pyspect import *
from pyspect.primitives import *
from pyspect.tlt import *
from pyspect.set_builder import *

TLT.select(ContLTL)


## 1) Create the 4x4 map


In [ ]:
# Map bounds: [0, 4] x [0, 4]
X0, XN = 0.0, 4.0
Y0, YN = 0.0, 4.0

dt = 0.2

# Obstacle 1: lower-left 2x2 square
lower_obs_x, lower_obs_y = 0.0, 0.0
lower_obs_w, lower_obs_h = 2.0, 2.0

# Obstacle 2: top corridor block, 2.8 long and 0.2 wide
upper_obs_x, upper_obs_y = 1.2, 3.0
upper_obs_w, upper_obs_h = 2.8, 0.2

# Goal region in the upper-right corner
goal_x, goal_y = 3.58, 3.46
goal_w, goal_h = 0.30, 0.36
target_set = BoundedSet(x=(goal_x, goal_x + goal_w), y=(goal_y, goal_y + goal_h))

# Checkpoint near the lower-right corner
checkpoint_pos = np.array([3.75, 0.28])
checkpoint_r = 0.12
checkpoint_set = BoundedSet(
    x=(checkpoint_pos[0] - checkpoint_r, checkpoint_pos[0] + checkpoint_r),
    y=(checkpoint_pos[1] - checkpoint_r, checkpoint_pos[1] + checkpoint_r),
)

# Car start: [x, y, yaw, delta, vel]
car_w, car_h = 0.25, 0.40
STATE_INIT = np.array([1.92, 2.34, pi, 0.0, 0.5])
print("STATE_INIT:", STATE_INIT)

# Keep map bounds slightly inside domain
map_box = BoundedSet(x=(X0 + 0.01, X0 + XN - 0.01), y=(Y0 + 0.01, Y0 + YN - 0.01))
lower_obstacle = AND(
    AND(map_box, BoundedSet(x=(..., lower_obs_x + lower_obs_w))),
    BoundedSet(y=(..., lower_obs_y + lower_obs_h)),
)
upper_obstacle = AND(
    AND(map_box, BoundedSet(x=(upper_obs_x, ...))),
    BoundedSet(y=(upper_obs_y, upper_obs_y + upper_obs_h)),
)
obstacles = OR(lower_obstacle, upper_obstacle)

# Forward being in direction of y
_v = .3 # moving obstacle speed
_w = .4 # moving obstacle width
_x = 2.0 # moving obstacle center x
moving_obstacle_boundary = BoundedSet(x=(_x-_w/2, _x+_w/2), y=(3., ...))
moving_obstacle = EMPTY
for i, t in enumerate([0.1, 5.1, 10.1, 15.1, 19.1]):
    __v = +_v if i % 2 == 0 else -_v
    __t = t + 0.5 if i % 2 == 0 else t - 0.5
    infront_moving_obstacle = HalfSpaceSet(normal=[ +1, -__v],
                                           offset=[3.5,    t],
                                           axes = ['y',  't'])

    behind_moving_obstacle = HalfSpaceSet(normal=[ -1, +__v],
                                          offset=[3.0,  __t],
                                          axes = ['y',  't'])
    moving_obstacle = Union(moving_obstacle, reduce(Inter, [
        Compl(infront_moving_obstacle), 
        Compl(behind_moving_obstacle),
        moving_obstacle_boundary,
    ]))

safe_region = AND(map_box, NOT(OR(obstacles, moving_obstacle)))

print(f"Lower obstacle: x=({lower_obs_x}, {lower_obs_x + lower_obs_w}), y=({lower_obs_y}, {lower_obs_y + lower_obs_h})")
print(f"Upper obstacle: x=({upper_obs_x}, {upper_obs_x + upper_obs_w}), y=({upper_obs_y}, {upper_obs_y + upper_obs_h})")
print(f"Checkpoint region: x=({checkpoint_pos[0] - checkpoint_r}, {checkpoint_pos[0] + checkpoint_r}), y=({checkpoint_pos[1] - checkpoint_r}, {checkpoint_pos[1] + checkpoint_r})")
print(f"Goal region: x=({goal_x}, {goal_x + goal_w}), y=({goal_y}, {goal_y + goal_h})")


In [ ]:
# impl.plot(obs := moving_obstacle(impl), axes=('x', 'y', 't'), colorscale='reds').show()
# print(obs.min(), obs.max())

In [ ]:
from matplotlib.patches import Rectangle
from matplotlib.transforms import Affine2D

fig, ax = plt.subplots(figsize=(7, 7))

# Workspace square
ax.add_patch(Rectangle((X0, Y0), XN, YN, fill=False, lw=2, edgecolor="black"))

# Obstacles
ax.add_patch(Rectangle((lower_obs_x, lower_obs_y), lower_obs_w, lower_obs_h, facecolor="white", edgecolor="tab:blue", hatch="++", linewidth=0.0, label="Obstacle"))
ax.add_patch(Rectangle((upper_obs_x, upper_obs_y), upper_obs_w, upper_obs_h, facecolor="white", edgecolor="tab:blue", hatch="////", linewidth=0.0))

# Goal set
ax.add_patch(Rectangle((goal_x, goal_y), goal_w, goal_h, facecolor="#34698d", edgecolor="#0c2d48", linewidth=1.8, label="Goal set"))
ax.scatter(checkpoint_pos[0], checkpoint_pos[1], marker="^", s=380, color="black", zorder=6)

# Car pose
heading_deg = np.degrees(STATE_INIT[2])
car_rect = Rectangle((-car_h / 2, -car_w / 2), car_h, car_w, facecolor="0.8", edgecolor="0.5", alpha=0.95, linewidth=1.0, label="Car")
t = Affine2D().rotate_deg(heading_deg).translate(STATE_INIT[0], STATE_INIT[1]) + ax.transData
car_rect.set_transform(t)
ax.add_patch(car_rect)
ax.plot(STATE_INIT[0], STATE_INIT[1], "ko", markersize=2.5)

# Figure annotations
ax.hlines(upper_obs_y, X0, upper_obs_x, colors="k", linestyles=(0, (2, 2)), linewidth=0.8, alpha=0.6)
ax.text(upper_obs_x + upper_obs_w / 2, upper_obs_y - 0.06, "2.8 length", ha="center", va="top", fontsize=14)
ax.text(upper_obs_x + upper_obs_w + 0.05, upper_obs_y + upper_obs_h / 2, "0.2 width", ha="left", va="center", fontsize=14)
ax.text(checkpoint_pos[0] + 0.24, checkpoint_pos[1], "Checkpoint", ha="left", va="center", fontsize=15)
ax.text(goal_x + goal_w + 0.14, goal_y + goal_h / 2, "Goal set", ha="left", va="center", fontsize=15)

ax.set_xlim(X0, X0 + XN)
ax.set_ylim(Y0, Y0 + YN)
ax.set_aspect("equal", adjustable="box")
ax.set_title("4x4 corridor map")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_xticks([0, 2, 4])
ax.set_yticks([2, 3, 4])
ax.grid(False)
plt.show()


## 2) Specify the task in Temporal Logic Tree form

In [ ]:
# Task specification (LTL):
#   (safe_region Until target_set) And ((safe_region And Not(target_set)) Until checkpoint_set)
#
# Direct AND of two UNTIL formulas cannot be realized as a single BRT intersection:
# the intersection of two independent reach tubes certifies two separate trajectories,
# not one trajectory satisfying both conjuncts simultaneously.
#
# Correct two-stage realization:
#   Stage 1  V1  =  reach(target_set,                   safe_region)
#   Stage 2  out =  reach(checkpoint ∩ {V1 ≤ 0},  safe_region ∧ ¬target_set)
#
# Stage 2 uses (safe ∧ ¬target) as the constraint, matching the second conjunct exactly.
# Any state certified by `out` has a SINGLE trajectory:
#   - stays in safe ∧ ¬target until hitting a checkpoint state from which target is reachable
#   - then continues to target while staying safe

## 3) Compute the BRT using TVHJImpl

In [ ]:
from pyspect.systems.hj_reachability.bicycle5d import Bicycle5D
from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.plotting.levelsets import plot3D_levelset

jax.config.update("jax_platform_name", "cpu")

t_max = 20

# Steering limits [rad] and acceleration limits [m/s²]
max_steer = 0.5
max_accel = 1.0
max_vel   = 1.5

impl = TVHJImpl(
    dynamics=dict(
        cls=Bicycle5D,
        min_steer=-max_steer,
        max_steer=max_steer,
        min_accel=-max_accel,
        max_accel=max_accel,
        wheelbase=0.32,
    ),
    axis_specs=[
        dict(name='t',      bounds=[0, t_max],           points=101),
        dict(name='x',      bounds=[X0, X0 + XN],        points=25),
        dict(name='y',      bounds=[Y0, Y0 + YN],        points=25),
        dict(name='*yaw',   bounds=[-pi, pi],             points=15),
        dict(name='delta',  bounds=[-max_steer, max_steer], points=11),
        dict(name='vel',    bounds=[0.0, max_vel],        points=11),
    ],
)

In [ ]:
impl.plot(obs := moving_obstacle(impl), axes=('x', 'y', 't'), colorscale='reds').show()
print(obs.min(), obs.max())

In [ ]:
# Execution order:  x0 --[safe ∧ ¬target]--> checkpoint --[safe]--> target
#
# Because BRT computation is backward in time, we must compute the LATER subtask first:
#
# Step 1 (computed first, later in execution): BRT for safe_region Until target_set
#   V_target(t,x) ≤ 0  ⟺  from (t,x), ∃ trajectory staying in safe until reaching target
V_target = TLT.construct(UNTIL(safe_region, target_set)).realize(impl)

# Step 2 (computed second, earlier in execution): BRT for (safe ∧ ¬target) Until (checkpoint ∩ {V_target ≤ 0})
#   out1(t,x) ≤ 0  ⟺  from (t,x), ∃ trajectory staying in safe ∧ ¬target
#                       until reaching a checkpoint state from which target is still reachable
checkpoint_then_target = Inter(checkpoint_set, Set(V_target))
safe_no_target = AND(safe_region, NOT(target_set))

out1 = TLT.construct(UNTIL(safe_no_target, checkpoint_then_target)).realize(impl)
print("BRT shape:", out1.shape)

np.save("out_bicycle5d.npy", np.asarray(out1))
print("Saved")

In [ ]:
impl.plot(
    out1.sum(axis=0, keepdims=True),
    method='contour',
    axes=('x', 'y'),
    # transform_select=[('t', 10.)],
)

In [ ]:
impl.plot(out1, axes=('x', 'y', 't'))